# Álgebra

  Para a base de álgebra, primeiramente foram importadas funções e constantes dos módulos constantes.py, aritmetica.py, funcoes.py e numeros_complexos.py e em alguns agoritmos foram importadas outras funções para uso momentâneo, desenvolvidos anteriormente. 
  
  Em seguida foi definida uma classe chmada "Polinomio", usada para representar os polinômios. Também foram definidas as maneiras com que o polinômio é interpretado internamente, como ele aparecerá ao ser printado, como aparecerá internamente no código e qual o seu grau. Além disso, foi definido como os polinômios interagem entre si e com escalares por meio das operações matemáticas já definida no Python, sendo adição, subtração, multiplicação, negação; ainda foi definido como os polinômios interagem com operações de cálculo derivada, integral indefinida e integral definida e como são definidas suas raízes.

  Para o auxílio em outros algoritmos, foram definidos algoritmos auxiliares para a avaliação de Horner; do polinômio de Newton; divisão sintética de polinômios, que no caso seria a divisão por um polinômio "x-r"; e para a resolução de polinômios do 2° grau.

  Afim de resolver equações, foram desenvolvidos algoritmos para resolver equações do 2° grau completas, resolver equações do 3° grau completas e para achar as raízes reais de um polinômio. Em seguida foram feitos algoritmos para algumas propriedades específicas dos polinômios, sendo a divisão polinomial, mdc polinomial, composição dos polinômios, composição de polinômios a partir de suas raízes, definição das raízes como objetos complexos, transformação de polinõmios em frações parciais lineares e verificação se é possível fatorar o polinômio usando coeficientes reais.

  Para finalizar, adicionou-se a linha de código "%%writefile algebra.py" para converter o código em um módulo exportável para ser usado em outros módulos e no uso final. Pelo mesmo motivo, o código teve que ser colocado inteiramente em uma única célula de código, pois o comando usado para isso exporta códigos de apenas uma célula para isso.

In [1]:
%%writefile algebra.py

from constantes import tolerancia_de_erro, maximo_de_interacoes, pi
from aritmetica import valor_absoluto, mdc, fatorial
from funcoes import sqrt, cbrt
from numeros_complexos import Complexo

#Definição de classe para polinômios
class Polinomio:

    def __init__(proprio, coeficientes: list):
        if not coeficientes:
            raise ValueError("Polinômio não pode ter coeficientes vazios.")
        while len(coeficientes) > 1 and coeficientes[0] == 0:
            coeficientes = coeficientes[1:]
        proprio.coeficientes = [float(c) for c in coeficientes]

    @property
    def grau(proprio) -> int:
        return len(proprio.coeficientes) - 1

    def __str__(proprio) -> str:
        termos = []
        n = proprio.grau
        for i, c in enumerate(proprio.coeficientes):
            exp = n - i
            if c == 0:
                continue
            if exp == 0:
                termos.append(f"{c:g}")
            elif exp == 1:
                termos.append(f"{c:g}x")
            else:
                termos.append(f"{c:g}x^{exp}")
        return " + ".join(termos).replace("+ -", "- ") or "0"

    def __repr__(proprio) -> str:
        return f"Polinomio({proprio.coeficientes})"

    def __call__(proprio, x: float) -> float:
        return avaliacao_horner(proprio.coeficientes, x)

    #Interpretação das operações do Python
    def __add__(proprio, outro: "Polinomio") -> "Polinomio":
        a, b = proprio.coeficientes[:], outro.coeficientes[:]
        while len(a) < len(b):
            a.insert(0, 0.0)
        while len(b) < len(a):
            b.insert(0, 0.0)
        return Polinomio([ai + bi for ai, bi in zip(a, b)])

    def __sub__(proprio, outro: "Polinomio") -> "Polinomio":
        a, b = proprio.coeficientes[:], outro.coeficientes[:]
        while len(a) < len(b):
            a.insert(0, 0.0)
        while len(b) < len(a):
            b.insert(0, 0.0)
        return Polinomio([ai - bi for ai, bi in zip(a, b)])

    def __mul__(proprio, outro) -> "Polinomio":
        if isinstance(outro, (int, float)):
            return Polinomio([c * outro for c in proprio.coeficientes])
        if isinstance(outro, Polinomio):
            m, n = proprio.grau, outro.grau
            resultado = [0.0] * (m + n + 1)
            for i, ai in enumerate(proprio.coeficientes):
                for j, bj in enumerate(outro.coeficientes):
                    resultado[i + j] += ai * bj
            return Polinomio(resultado)
        return NotImplemented

    def __rmul__(proprio, outro):
        return proprio.__mul__(outro)

    def __neg__(proprio) -> "Polinomio":
        return Polinomio([-c for c in proprio.coeficientes])

    def __eq__(proprio, outro) -> bool:
        if not isinstance(outro, Polinomio):
            return False
        a, b = proprio.coeficientes[:], outro.coeficientes[:]
        while len(a) < len(b):
            a.insert(0, 0.0)
        while len(b) < len(a):
            b.insert(0, 0.0)
        return all(valor_absoluto(ai - bi) < tolerancia_de_erro for ai, bi in zip(a, b))

    #Interpretação das operações do módulo de cálculo
    def derivada(proprio) -> "Polinomio":
        if proprio.grau == 0:
            return Polinomio([0.0])
        n = proprio.grau
        novos = []
        for i, c in enumerate(proprio.coeficientes[:-1]):
            exp = n - i
            novos.append(c * exp)
        return Polinomio(novos)

    def integral_indefinida(proprio, constante: float = 0.0) -> "Polinomio":
        n = proprio.grau
        novos = []
        for i, c in enumerate(proprio.coeficientes):
            exp = n - i + 1
            novos.append(c / exp)
        novos.append(constante)
        return Polinomio(novos)

    def integral_definida(proprio, a: float, b: float) -> float:
        P = proprio.integral_indefinida()
        return P(b) - P(a)

    def raizes(proprio, tolerancia: float = None) -> list:
        if tolerancia is None:
            tolerancia = tolerancia_de_erro
        if proprio.grau == 0:
            return []
        if proprio.grau == 1:
            return [-proprio.coeficientes[1] / proprio.coeficientes[0]]
        if proprio.grau == 2:
            a, b, c = proprio.coeficientes[0], proprio.coeficientes[1], proprio.coeficientes[2]
            return raizes_quadraticas(a, b, c)

        p = Polinomio(proprio.coeficientes[:])
        encontradas = []

        while p.grau >= 1:
            raiz = None
            for x0 in [-5.0, -2.0, -1.0, 0.0, 0.5, 1.0, 2.0, 5.0, 10.0]:
                try:
                    r = newton_polinomio(p, x0, tolerancia)
                    if r is not None:
                        raiz = r
                        break
                except Exception:
                    continue

            if raiz is None:
                break

            encontradas.append(round(raiz, 10))
            p = divisao_sintetica(p, raiz)

        return encontradas


#Algoritmos auxiliares
#Avaliação de Horner
def avaliacao_horner(coeficientes: list, x: float) -> float:
    resultado = 0.0
    for c in coeficientes:
        resultado = resultado * x + c
    return resultado


#Polinômio de Newton
def newton_polinomio(p: Polinomio, x0: float, tolerancia: float) -> float:
    dp = p.derivada()
    x = x0
    for _ in range(maximo_de_interacoes):
        fx = p(x)
        dfx = dp(x)
        if valor_absoluto(dfx) < tolerancia:
            return None
        x_novo = x - fx / dfx
        if valor_absoluto(x_novo - x) < tolerancia:
            return x_novo
        x = x_novo
    return x


#Divisão sintética (por (x - r))
def divisao_sintetica(p: Polinomio, r: float) -> Polinomio:
    coeficientes = p.coeficientes
    novos = [coeficientes[0]]
    for i in range(1, len(coeficientes) - 1):
        novos.append(coeficientes[i] + r * novos[-1])
    return Polinomio(novos)


#Resolução de polinômios do 2° grau
def raizes_quadraticas(a: float, b: float, c: float) -> list:
    delta = b * b - 4 * a * c
    if delta < 0:
        return []
    raiz_delta = sqrt(delta)
    return [(-b + raiz_delta) / (2 * a), (-b - raiz_delta) / (2 * a)]


#Resolução de equações
#Resolução de equação quadrática completa pela fórmula de Bhaskara
def bhaskara(a: float, b: float, c: float) -> dict:
    if a == 0:
        raise ValueError("Coeficiente 'a' não pode ser zero (não é equação quadrática).")

    delta = b * b - 4.0 * a * c
    resultado = {"discriminante": delta, "a": a, "b": b, "c": c}

    if valor_absoluto(delta) < tolerancia_de_erro:
        x = -b / (2 * a)
        resultado["tipo"] = "raiz_dupla"
        resultado["raizes"] = [x, x]
        resultado["descricao"] = f"Raiz real dupla: x = {x:.6f}"

    elif delta > 0:
        raiz_delta = sqrt(delta)
        x1 = (-b + raiz_delta) / (2 * a)
        x2 = (-b - raiz_delta) / (2 * a)
        resultado["tipo"] = "real"
        resultado["raizes"] = [x1, x2]
        resultado["descricao"] = f"Duas raízes reais: x_1 = {x1:.6f}, x_2 = {x2:.6f}"

    else:
        parte_real = -b / (2 * a)
        parte_imag = sqrt(-delta) / (2 * a)
        z1 = Complexo(parte_real, parte_imag)
        z2 = Complexo(parte_real, -parte_imag)
        resultado["tipo"] = "complexo"
        resultado["raizes"] = [z1, z2]
        resultado["descricao"] = f"Raízes complexas conjugadas: x_1 = {z1}, x_2 = {z2}"

    return resultado


#Resolução de equação cúbica pela fórmula de Cardano
def formula_cardano(a: float, b: float, c: float, d: float) -> dict:
    from trigonometria import cos, arccos

    if a == 0:
        raise ValueError("Coeficiente 'a' não pode ser zero (não é equação cúbica).")

    p_coeficiente = (3.0 * a * c - b * b) / (3.0 * a * a)
    q_coeficiente = (2.0 * b ** 3 - 9.0 * a * b * c + 27.0 * a * a * d) / (27.0 * a ** 3)
    deslocamento = b / (3.0 * a)

    discriminante = (q_coeficiente / 2.0) ** 2 + (p_coeficiente / 3.0) ** 3
    resultado = {"discriminante": discriminante, "a": a, "b": b, "c": c, "d": d}

    if discriminante > tolerancia_de_erro:
        raiz_discriminante = sqrt(discriminante)
        u = cbrt(-q_coeficiente / 2.0 + raiz_discriminante)
        v = cbrt(-q_coeficiente / 2.0 - raiz_discriminante)
        t1 = u + v
        x1 = t1 - deslocamento
        parte_real = -t1 / 2.0 - deslocamento
        parte_imag = (u - v) * sqrt(3.0) / 2.0

        resultado["tipo"] = "uma real e duas complexas"
        resultado["raizes"] = [
            Complexo(x1, 0.0),
            Complexo(parte_real, parte_imag),
            Complexo(parte_real, -parte_imag),
        ]

    elif valor_absoluto(discriminante) <= tolerancia_de_erro:
        u = cbrt(-q_coeficiente / 2.0)
        t1 = 2.0 * u
        t2 = -u
        x1 = t1 - deslocamento
        x2 = t2 - deslocamento

        resultado["tipo"] = "real com repeticao"
        resultado["raizes"] = [
            Complexo(x1, 0.0),
            Complexo(x2, 0.0),
            Complexo(x2, 0.0),
        ]

    else:
        raiz_termo = sqrt(-p_coeficiente / 3.0)
        argumento = (3.0 * q_coeficiente) / (2.0 * p_coeficiente * raiz_termo)
        if argumento > 1.0:
            argumento = 1.0
        if argumento < -1.0:
            argumento = -1.0
        theta = arccos(argumento)

        raizes_t = []
        for k in range(3):
            t_k = 2.0 * raiz_termo * cos((theta - 2.0 * pi * k) / 3.0)
            raizes_t.append(t_k - deslocamento)

        resultado["tipo"] = "tres reais distintas"
        resultado["raizes"] = [Complexo(r, 0.0) for r in raizes_t]

    return resultado


#Definição das raízes racionais
def raizes_racionais(p: Polinomio) -> list:
    c0 = int(valor_absoluto(p.coeficientes[-1]))
    cn = int(valor_absoluto(p.coeficientes[0]))

    if c0 == 0:
        return [0.0] + raizes_racionais(divisao_sintetica(p, 0.0))

    def divisores(k):
        k = abs(int(k))
        return [i for i in range(1, k + 1) if k % i == 0]

    divisores_c0 = divisores(c0)
    divisores_cn = divisores(cn)

    raizes = []
    vistas = set()
    for numerador in divisores_c0:
        for denominador in divisores_cn:
            g = mdc(numerador, denominador)
            r_positivo = (numerador // g) / (denominador // g)
            r_negativo = -r_positivo
            for r in (r_positivo, r_negativo):
                chave = round(r, 12)
                if chave not in vistas and valor_absoluto(p(r)) < tolerancia_de_erro:
                    raizes.append(r)
                    vistas.add(chave)
    return raizes


#Divisão polinomial
def divisao_polinomial(p: Polinomio, q: Polinomio) -> tuple:
    if q.coeficientes == [0.0]:
        raise ValueError("Divisão por polinômio nulo.")

    resto = Polinomio(p.coeficientes[:])
    grau_q = q.grau
    coeficiente_lider_q = q.coeficientes[0]

    coeficientes_quociente = []

    while resto.grau >= grau_q and resto.coeficientes != [0.0]:
        diferenca_grau = resto.grau - grau_q
        fator = resto.coeficientes[0] / coeficiente_lider_q
        coeficientes_quociente.append(fator)

        termo = [fator * c for c in q.coeficientes] + [0.0] * diferenca_grau
        while len(termo) < len(resto.coeficientes):
            termo.insert(0, 0.0)

        novos_coeficientes = [resto.coeficientes[i] - termo[i] for i in range(len(resto.coeficientes))]
        resto = Polinomio(novos_coeficientes)

        if resto.grau == diferenca_grau - 1 and diferenca_grau > 1:
            coeficientes_quociente.extend([0.0] * (diferenca_grau - 1))

    if not coeficientes_quociente:
        coeficientes_quociente = [0.0]

    return Polinomio(coeficientes_quociente), resto


#MDC polinomial
def mdc_polinomial(p: Polinomio, q: Polinomio) -> Polinomio:
    a, b = p, q
    while b.coeficientes != [0.0]:
        _, resto = divisao_polinomial(a, b)
        a, b = b, resto

    if a.coeficientes[0] != 0:
        normalizado = [c / a.coeficientes[0] for c in a.coeficientes]
        return Polinomio(normalizado)
    return a


#Composição polinomial
def compor_polinomios(p: Polinomio, q: Polinomio) -> Polinomio:
    resultado = Polinomio([0.0])
    for c in p.coeficientes:
        resultado = resultado * q + Polinomio([c])
    return resultado


#Construção de polinômios a partir de suas raízes
def polinomio_a_partir_de_raizes(raizes: list) -> Polinomio:
    resultado = Polinomio([1.0])
    for r in raizes:
        if isinstance(r, Complexo):
            fator = Polinomio([1.0, -r.real])
        else:
            fator = Polinomio([1.0, -float(r)])
        resultado = resultado * fator
    return resultado


#Definição de raízes como objetos complexos
def raizes_complexas(p: Polinomio) -> list:
    if p.grau == 1:
        a, b = p.coeficientes[0], p.coeficientes[1]
        return [Complexo(-b / a, 0.0)]

    if p.grau == 2:
        a, b, c = p.coeficientes[0], p.coeficientes[1], p.coeficientes[2]
        resultado = bhaskara(a, b, c)
        if resultado["tipo"] == "complexo":
            return resultado["raizes"]
        return [Complexo(r, 0.0) for r in resultado["raizes"]]

    raise ValueError(f"raizes_complexas requer grau 1 ou 2, recebido grau {p.grau}.")


#Decomposição de polinômios em frações parciais
def fracoes_parciais_lineares(p: Polinomio, raizes_q: list) -> list:
    n = len(raizes_q)
    for i in range(n):
        for j in range(i + 1, n):
            if valor_absoluto(raizes_q[i] - raizes_q[j]) < tolerancia_de_erro:
                raise ValueError("fracoes_parciais_lineares requer raízes distintas; ""raízes repetidas exigem termos de potência maior.")

    coeficientes = []
    for i in range(n):
        ri = raizes_q[i]
        numerador = avaliacao_horner(p.coeficientes, ri)
        denominador = 1.0
        for j in range(n):
            if j != i:
                denominador *= (ri - raizes_q[j])
        coeficientes.append(numerador / denominador)

    return coeficientes


#Verificação se um polinômio não pode ser fatorado usando coeficientes reais
def eh_irredutivel_racional(p: Polinomio) -> bool:
    if p.grau not in (2, 3):
        raise ValueError(f"eh_irredutivel_racional só é válido para grau 2 ou 3.")

    raizes = raizes_racionais(p)
    return len(raizes) == 0

Overwriting algebra.py
